# 02 - Fit, diagnose, and choose between variants

Run `make data` and `make model` first.  This notebook is where you decide what
the model is allowed to keep.

In [ ]:
import sys
from pathlib import Path

# work whether the kernel starts in the repo root or in notebooks/
root = next((p.parent for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ecpmodel').is_dir()), Path.cwd())
sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ecpmodel.features.build import REL_MULT, build_panel
from ecpmodel.io import load_cfg, load_interim
from ecpmodel.models.draws import param_draws, state_draws
from ecpmodel.models.ecm_ssm import fit, summarize
from ecpmodel.models.real import deflator, real_draws

pd.options.display.width = 120
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
panel = build_panel(horizon=3)
mod, res = fit(panel, obs=['headline', 'comext'], quiet=False)
print(res.summary())

## 1. Parameter table

Read the signs before anything else.  Every coefficient on the cost block should
be positive (higher feedstock cost, higher contract price); `gamma` must be
negative (the price closes the gap) and `theta` in [0, 1] (partial same-month
pass-through).  `gamma` is weakly identified in short samples - report its
standard error and do not interpret a half-life without one.

In [ ]:
params = pd.Series(np.asarray(res.params), index=mod.param_names, name='coef')
se = pd.Series(np.sqrt(np.diag(np.asarray(res.cov_params()))), index=mod.param_names,
               name='std_err')
table = pd.concat([params, se], axis=1)
table['t'] = table['coef'] / table['std_err'].replace(0, np.nan)
table

## 2. Standardised residuals

You want roughly unit variance, no obvious autocorrelation and no trending
bias.  Run the Ljung-Box test per observation equation.

In [ ]:
fig, axes = plt.subplots(len(mod.obs_names), 1, figsize=(11, 2.3 * len(mod.obs_names)),
                         sharex=True)
axes = np.atleast_1d(axes)
idx = mod.data.row_labels
err = np.asarray(res.standardized_forecasts_error).reshape(len(mod.obs_names), -1)
for ax, i, name in zip(axes, range(len(mod.obs_names)), mod.obs_names):
    e = err[i]
    ax.plot(idx, e, lw=.8, color='#12395b')
    ax.axhline(0, color='k', lw=.6)
    ax.axhline(2, color='#a33b3b', lw=.6, ls=':')
    ax.axhline(-2, color='#a33b3b', lw=.6, ls=':')
    ax.set_title(f'{name}: standardised residuals', fontsize=9)

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# statsmodels' built-in test is univariate-only, so run it per observation
err = np.asarray(res.standardized_forecasts_error).reshape(len(mod.obs_names), -1)
for i, name in enumerate(mod.obs_names):
    e = err[i][~np.isnan(err[i])]
    if len(e) < 30:
        print(f'{name:10s} too few residuals ({len(e)}) to test')
        continue
    q = acorr_ljungbox(e, lags=[6, 12], return_df=True)
    print(f'{name:10s} Ljung-Box  h=6: Q={q.loc[6, "lb_stat"]:8.2f} p={q.loc[6, "lb_pvalue"]:.4f}'
          f'   h=12: Q={q.loc[12, "lb_stat"]:8.2f} p={q.loc[12, "lb_pvalue"]:.4f}')

## 3. Smoothed ECP versus the cost anchor

The gap is the implied cracker margin.  It should look economically sensible -
compressed in downturns, wide in tight markets.  If it sits permanently far
positive or negative, the intercept or `a_c` is soaking up a basis difference
rather than a real margin.

The first `k_states - 1` months are diffuse-start burn-in; hide them.

In [ ]:
out = summarize(res, panel)
live = out[~out['burn_in']]

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.fill_between(live.index, live['lo80'], live['hi80'], color='#7f9fbf', alpha=.5,
                label='80% band')
ax.plot(live.index, live['ecp_mean'], color='#12395b', lw=1.8, label='latent ECP')
ax.plot(live.index, live['cost_anchor'], color='#c1666b', lw=1.1, ls='--',
        label="cost anchor (beta'x)")
ax.set_ylabel('EUR / t')
ax.legend(frameon=False, fontsize=8)
ax.set_title('Latent ECP and the estimated cost anchor')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.fill_between(live.index, 0, live['margin'],
                where=live['margin'] >= 0, color='#3f7d4e', alpha=.5)
ax.fill_between(live.index, 0, live['margin'],
                where=live['margin'] < 0, color='#a33b3b', alpha=.5)
ax.axhline(0, color='k', lw=.8)
ax.set_title('Implied margin (latent ECP - cost anchor), EUR/t')

## 4. Variant comparison - in-sample log-likelihood

Likelihood comparison only *proposes*; out-of-sample CRPS decides.  The
question each row answers:

* **lag weights**: pick among 3-5 shapes, never a continuous grid.
* **comext lag**: 1 vs 2 (a lag-2 Comext needs a deeper state).
* **PPI on/off**: the PPI is a weak instrument - if turning it on makes
  `gamma` collapse to zero and `b_p` explode, it has taken over the price.
* **regime on/off**: is `kappa` significantly above zero, and does it help
  out of sample?

In [ ]:
rows = []
for w in load_cfg('model')['lag_weights']:
    panel_w = build_panel(horizon=0, lag_w=w)
    try:
        m, r = fit(panel_w, n_starts=2, obs=['headline', 'comext'])
        rows.append({'variant': f'lag_w={tuple(w)}', 'llf': r.llf,
                     'gamma': r.params[m.layout.idx['gamma']],
                     'n_headlines': int(panel_w['endog']['headline'].notna().sum())})
    except Exception as exc:
        rows.append({'variant': f'lag_w={tuple(w)}', 'llf': np.nan,
                     'gamma': np.nan, 'n_headlines': 0, 'error': str(exc)})
pd.DataFrame(rows).round(3)

In [ ]:
rows = []
for label, obs in [('headline only', ['headline']),
                   ('headline+comext', ['headline', 'comext']),
                   ('headline+comext+ppi', ['headline', 'comext', 'ppi'])]:
    try:
        m, r = fit(panel, n_starts=3, obs=obs)
        rows.append({'variant': label, 'llf': r.llf,
                     'gamma': r.params[m.layout.idx['gamma']],
                     'theta': r.params[m.layout.idx['theta']],
                     'b_p': r.params[m.layout.idx['b_p']] if 'b_p' in m.layout.idx else np.nan,
                     's_p': r.params[m.layout.idx['s_p']] if 's_p' in m.layout.idx else np.nan})
    except Exception as exc:
        rows.append({'variant': label, 'llf': np.nan, 'error': str(exc)})
pd.DataFrame(rows).round(4)

**Reading the PPI row.**  If `b_p` is large (several k EUR/t per unit of the
standardised index) and `s_p` is tiny, the index has taken the place of the
price: the filter is no longer estimating a contract price, it is reporting a
chemicals PPI.  That is the failure mode the tutorial warns about, and it is
why the shipped default leaves PPI switched off.

## 5. Posterior draws: bands that include parameter uncertainty

The smoother bands are conditional on the fitted parameters.  Draw parameters
from their asymptotic normal and run the simulation smoother for each draw.

In [ ]:
draws = param_draws(res, n=200, seed=0)
paths = state_draws(mod, draws, seed=0, results=res)          # (n_draws, nobs), k EUR/t

q = np.quantile(paths * 1000.0, [0.05, 0.5, 0.95], axis=0)
draws_df = pd.DataFrame({'p05': q[0], 'p50': q[1], 'p95': q[2]}, index=idx)
draws_df.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.fill_between(live.index, live['lo90'], live['hi90'], color='#d9d9d9', alpha=.8,
                label='90%, smoother only')
ax.fill_between(live.index, draws_df['p05'].reindex(live.index),
                draws_df['p95'].reindex(live.index), color='#7f9fbf', alpha=.6,
                label='90% incl. parameter uncertainty')
ax.plot(live.index, live['ecp_mean'], color='#12395b', lw=1.8, label='smoother mean')
ax.plot(draws_df.index, draws_df['p50'], color='#c1666b', lw=1.0, ls='--',
        label='draws median')
ax.legend(frameon=False, fontsize=8)
ax.set_ylabel('EUR / t')
ax.set_title('Smoother bands vs simulation-smoother draws')

## 6. Real price

Two definitions, different questions: the PPI deflator is a purchasing-power
view, the Brent ratio is a margin view.  For forecast months the deflator is
held at its last value - document that, because the bands then ignore
inflation risk.

In [ ]:
ppi_files = sorted(Path('data/interim').glob('ppi_*.parquet'))
ppi = pd.read_parquet(ppi_files[0]).iloc[:, 0] if ppi_files else None

if ppi is not None and len(ppi):
    base = ppi.index[0]
    d = deflator(ppi, idx, base)
    real_draw = real_draws(paths, idx, ppi, base)
    rq = np.quantile(real_draw, [0.05, 0.5, 0.95], axis=0)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(idx, rq[1], color='#12395b', lw=1.6, label='real ECP, median')
    ax.fill_between(idx, rq[0], rq[2], color='#7f9fbf', alpha=.4, label='90% band')
    ax.set_ylabel(f'EUR / t, deflated by NACE C20 PPI (base {base:%Y-%m})')
    ax.legend(frameon=False, fontsize=8)
else:
    print('no PPI series - run `make data` first')

## 7. Before you publish

* `make backtest` and report `n` next to every number.
* Compare against the naive and cost-plus-OLS benchmarks at h=0 and h=1.
* Check 80% coverage by horizon, and separately over the 2021-22 window.
* Snapshot the raw pulls every month (`make data` writes dated files) - Comext
  revisions are the biggest silent source of overstatement.
* State plainly what this is: a *reference* price for central Europe inferred
  from public proxies. It is not any single buyer's contract; collars, rebates,
  indexation formulas and delivery basis are invisible in public data.